In [1]:
## Import modules
import os,sys
import numpy as np
import cftime 
import json 
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger 
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# Import utilities for this comparison
sys.path.insert(0,cmct_dir)
from cmct.time_utils import check_datarange
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.json_to_netcdf import *

from cmct.gravimetry import *
from cmct.projection import *


In [2]:
%matplotlib inline
#

In [3]:
# Ice sheet
loc = 'GIS' # 'GIS' or 'AIS'

# Set time range for comparison
start_year = 1975
end_year = 2015

# Set the observation data dir path
obs_filename = cmct_dir + '/data/calving/observed_icemask_ismip_annual.nc'


# Set the Model Data dir path
model_filename = cmct_dir + "/test/calving/sftgif_GIS_JPL_ISSM_historical.nc"

# Output filetype and filename
filetype = 'netcdf' # netcdf or json or None
filename = 'calving_comparison'


print(obs_filename)
gsfc = load_gsfc_calving(obs_filename)

print(model_filename)
model_res = load_model_calving(model_filename)
print(type(model_res))
# print(gsfc.print_info())  # Print the gsfc object to see its contents
print(vars(model_res)) 
  # Print the attributes of the gsfc object
  
  

/Users/aditya_pachpande/Documents/GitHub/CmCt/data/calving/observed_icemask_ismip_annual.nc
/Users/aditya_pachpande/Documents/GitHub/CmCt/test/calving/sftgif_GIS_JPL_ISSM_historical.nc
<class 'cmct.calving.Modelcalving'>
{'ds': <xarray.Dataset> Size: 72MB
Dimensions:   (y: 577, x: 337, nv4: 4, time: 36)
Coordinates:
    lon       (y, x) float64 2MB ...
    lat       (y, x) float64 2MB ...
  * time      (time) object 288B 1980-01-01 00:00:00 ... 2015-01-01 00:00:00
  * x         (x) float64 3kB -7.2e+05 -7.15e+05 -7.1e+05 ... 9.55e+05 9.6e+05
  * y         (y) float64 5kB -3.45e+06 -3.445e+06 ... -5.75e+05 -5.7e+05
Dimensions without coordinates: nv4
Data variables:
    lon_bnds  (y, x, nv4) float64 6MB ...
    lat_bnds  (y, x, nv4) float64 6MB ...
    ice_mask  (time, y, x) float64 56MB ...
Attributes:
    Conventions:  CF-1.6
    title:        ISMIP6 Projections Greenland model output
    institution:  NASA Jet Propulsion Laboratory, Pasadena, USA
    source:       ISSM
    references

In [4]:
# Simplifying date data type
gsfc.ds["time"] = gsfc.time

model_res.ds["time"] = [dt.year for dt in model_res.time.values]

# Handelling Time Range
# check_data_daterange(gsfc.time.values, model_res.time.values, start_year, end_year)


Choosing a range of x and y 
-

In [5]:
def cell_indice_to_plot(x, y):
    coor_x = ((x-1)* 1000) - 720000
    coor_y = ((y-1)* 1000) - 3450000
    return (coor_x, coor_y)

def upscaled_cell_indice_to_plot(x, y):
    coor_x = ((x-1) * 5000) - 720000
    coor_y = ((y-1) * 5000) - 3450000
    return (coor_x, coor_y)

In [10]:
def interpolate(gsfc, model_res, method):
    resampler = Resampling(gsfc, model_res, 'n')
    return resampler.resample()

    # Update the gsfc object with the resampled data
new_gsfc = interpolate(gsfc, model_res, 'linear')

# Save the resampled data to a new NetCDF file
# output_netcdf_filename = cmct_dir + '/notebooks/Calving/resampled_calving_obs_data.nc'
# resampled_data.to_netcdf(output_netcdf_filename, mode='w', format='NETCDF4')

In [ ]:
# Create 2 subplots side by side to compare the region 

def plot_gsfc_comparison(year=2006, area = 1):
    """Compare original GSFC vs resampled GSFC in specified coordinate regions"""
    
    if area == 1:
        top_left = (1165, 1830)
        top_right = (1312, 1830)
        bottom_left = (1165, 1578)
        bottom_right = (1312, 1578)

    # UPSCALED
        upscaled_top_left = (233, 366)
        upscaled_top_right = (262, 366)
        upscaled_bottom_left = (233, 316)
        upscaled_bottom_right = (262, 316)
            
    elif area == 2:
        # ORIGINAL
        top_left = (815, 543)
        top_right = (887, 543)
        bottom_left = (815, 435)
        bottom_right = (887, 435)

        # UPSCALED
        upscaled_top_left = (163, 109)
        upscaled_top_right = (178, 109)
        upscaled_bottom_left = (163, 87)
        upscaled_bottom_right = (178, 87)
    
    elif area == 3:
        # ORIGINAL
        top_left       = (643, 381)
        top_right      = (883, 381)
        bottom_left    = (643, 137)
        bottom_right   = (883, 137)

        # UPSCALED
        upscaled_top_left     = (129, 77)
        upscaled_top_right    = (177, 77)
        upscaled_bottom_left  = (129, 28)
        upscaled_bottom_right = (177, 28)
        
    elif area == 4:
        top_left       = (417, 1595)
        top_right      = (515, 1595)
        bottom_left    = (417, 1426)
        bottom_right   = (515, 1426)

        # UPSCALED 5-km grid
        upscaled_top_left     = (84, 319)
        upscaled_top_right    = (103, 319)
        upscaled_bottom_left  = (84, 286)
        upscaled_bottom_right = (103, 286)
        
    elif area == 5:
        # 1-km grid (native resolution)
        top_left       = (288, 2025)
        top_right      = (435, 2025)
        bottom_left    = (288, 1772)
        bottom_right   = (435, 1772)

        # UPSCALED 5-km grid
        upscaled_top_left     = (58, 405)
        upscaled_top_right    = (87, 405)
        upscaled_bottom_left  = (58, 355)
        upscaled_bottom_right = (87, 355)
        

    # Get data for the specified year
    gsfc_data = gsfc.ds.sel(year=year).ice_mask
    new_gsfc_data = new_gsfc.sel(year=year).ice_mask

    # Calculate coordinate bounds for original region (using 1km resolution)
    orig_x_min = cell_indice_to_plot(top_left[0], top_left[1])[0]
    orig_x_max = cell_indice_to_plot(top_right[0], top_right[1])[0]
    orig_y_min = cell_indice_to_plot(bottom_left[0], bottom_left[1])[1]
    orig_y_max = cell_indice_to_plot(top_left[0], top_left[1])[1]
    
    # Calculate coordinate bounds for upscaled region (using 5km resolution)
    upsc_x_min = upscaled_cell_indice_to_plot(upscaled_top_left[0], upscaled_top_left[1])[0]
    upsc_x_max = upscaled_cell_indice_to_plot(upscaled_top_right[0], upscaled_top_right[1])[0]
    upsc_y_min = upscaled_cell_indice_to_plot(upscaled_bottom_left[0], upscaled_bottom_left[1])[1]
    upsc_y_max = upscaled_cell_indice_to_plot(upscaled_top_left[0], upscaled_top_left[1])[1]
    
    # Create subplot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 12))
    
    # Plot 1: Original GSFC data with original coordinates
    gsfc_cropped = gsfc_data.sel(x=slice(orig_x_min, orig_x_max), y=slice(orig_y_min, orig_y_max))
    im1 = ax1.imshow(gsfc_cropped, 
                     extent=[orig_x_min, orig_x_max, orig_y_min, orig_y_max],
                     origin='lower', cmap='viridis', aspect='auto')
    ax1.set_title(f'Original GSFC Data (1km)\nYear: {year}')
    ax1.set_xlabel('X coordinate (m)')
    ax1.set_ylabel('Y coordinate (m)')
    
    # Add rectangle showing the exact region
    from matplotlib.patches import Rectangle
    rect1 = Rectangle((orig_x_min, orig_y_min), 
                     orig_x_max - orig_x_min, 
                     orig_y_max - orig_y_min,
                     linewidth=2, edgecolor='red', facecolor='none')
    ax1.add_patch(rect1)
    
    # Plot 2: Resampled GSFC data with upscaled coordinates  
    new_gsfc_cropped = new_gsfc_data.sel(x=slice(upsc_x_min, upsc_x_max), y=slice(upsc_y_min, upsc_y_max))
    im2 = ax2.imshow(new_gsfc_cropped,
                     extent=[upsc_x_min, upsc_x_max, upsc_y_min, upsc_y_max], 
                     origin='lower', cmap='viridis', aspect='auto')
    ax2.set_title(f'Resampled GSFC Data (5km)\nYear: {year}')
    ax2.set_xlabel('X coordinate (m)')
    ax2.set_ylabel('Y coordinate (m)')
    
    # Add rectangle showing the exact upscaled region
    rect2 = Rectangle((upsc_x_min, upsc_y_min),
                     upsc_x_max - upsc_x_min,
                     upsc_y_max - upsc_y_min, 
                     linewidth=2, edgecolor='blue', facecolor='none')
    ax2.add_patch(rect2)
    
    # Add colorbars
    plt.colorbar(im1, ax=ax1, label='Ice Mask Value')
    plt.colorbar(im2, ax=ax2, label='Ice Mask Value')
    
    # # Print statistics
    # print(f"\nYear {year} Statistics:")
    # print(f"Original region shape: {gsfc_cropped.shape}")
    # print(f"Original region range: {float(gsfc_cropped.min()):.3f} to {float(gsfc_cropped.max()):.3f}")
    # print(f"Resampled region shape: {new_gsfc_cropped.shape}")
    # print(f"Resampled region range: {float(new_gsfc_cropped.min()):.3f} to {float(new_gsfc_cropped.max()):.3f}")
    
    plt.tight_layout()
    plt.show()

# Test the function
try:
    print("Testing comparison plot...")
    # plot_gsfc_comparison(2006)
    
    # Create interactive widget
    print("\nCreating interactive widget...")
    time_slider = FloatSlider(
        value=start_year,
        min=start_year, 
        max=end_year,
        step=1,
        description='Year:',
        continuous_update=True
    )
    
    area_slider = FloatSlider(
        value=3,
        min=1, 
        max=5,
        step=1,
        description='Area:',
        continuous_update=True
    )
    
    interact(plot_gsfc_comparison, year=time_slider, area=area_slider)
    
except Exception as e:
    print(f"Error in plotting: {e}")
    print("Checking data availability...")
    
    # Debug information
    print(f"gsfc.ds type: {type(gsfc.ds)}")
    print(f"new_gsfc type: {type(new_gsfc)}")
    
    if hasattr(gsfc.ds, 'ice_mask'):
        print(f"gsfc.ds.ice_mask shape: {gsfc.ds.ice_mask.shape}")
    if hasattr(new_gsfc, 'ice_mask'):
        print(f"new_gsfc.ice_mask shape: {new_gsfc.ice_mask.shape}")
    
    # Check coordinate ranges
    print(f"\nOriginal coordinate bounds:")
    print(f"X: {cell_indice_to_plot(top_left[0], top_left[1])[0]} to {cell_indice_to_plot(top_right[0], top_right[1])[0]}")
    print(f"Y: {cell_indice_to_plot(bottom_left[0], bottom_left[1])[1]} to {cell_indice_to_plot(top_left[0], top_left[1])[1]}")
    
    print(f"\nUpscaled coordinate bounds:")
    print(f"X: {upscaled_cell_indice_to_plot(upscaled_top_left[0], upscaled_top_left[1])[0]} to {upscaled_cell_indice_to_plot(upscaled_top_right[0], upscaled_top_right[1])[0]}")
    print(f"Y: {upscaled_cell_indice_to_plot(upscaled_bottom_left[0], upscaled_bottom_left[1])[1]} to {upscaled_cell_indice_to_plot(upscaled_top_left[0], upscaled_top_left[1])[1]}")

Testing comparison plot...

Creating interactive widget...


interactive(children=(FloatSlider(value=1975.0, description='Year:', max=2015.0, min=1975.0, step=1.0), FloatS…

In [8]:
import logging.config
logging.config.dictConfig({
    'version': 1,
    'disable_existing_loggers': True,
})
for i in ['linear', 'nearest', 'makima', 'pchip', 'cubic', 'lanczos3d']:
    print(i)
    new_gsfc = interpolate(gsfc, model_res, i)
    plot_gsfc_comparison(2006, 2)

linear


KeyError: "'year' is not a valid dimension or coordinate for Dataset with dimensions FrozenMappingWarningOnValuesAccess({'y': 2880, 'x': 1680, 'nv4': 4, 'time': 36})"